# BoTorch qNEHVI — Multi-Objective Optimization for Image Classification

**MOML Assignment** — Prof. Aswin Kannan · Niranjan Gopal, Divyam Sareen · Due 2026-04-28

Optimizes 3 conflicting objectives over a CNN search space using **BoTorch qNEHVI** (q-Log-Noisy Expected Hypervolume Improvement, Bayesian / GP-driven):

1. **Maximize** classification accuracy (top-1 on test set)
2. **Minimize** inference time (ms / sample, CPU, batch=1)
3. **Minimize** model size (trainable parameters)

Hard wall-clock cap: **≤ 1.5 hours**, designed for running concurrently with the pymoo notebook on the same GPU.

qNEHVI fits a `SingleTaskGP` over the observed `(config, objective_vector)` pairs, then chooses each next config by maximizing expected hypervolume improvement under the GP posterior. The 10-D unit-cube encoding is decoded to the **same** discrete/continuous schema as the pymoo notebook, so the two studies are directly comparable. See the pymoo notebook for the head-to-head.

In [1]:
# --- Imports & path setup -------------------------------------------------
import sys, os, json
from pathlib import Path

# This notebook lives in notebooks/; make src/ importable.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

from data_loader import DEVICE, PROJECT_ROOT, get_dataset_info
from moo_botorch import run_botorch_study
from analyze_study import analyze

print(f"Device       : {DEVICE}")
print(f"Project root : {PROJECT_ROOT}")
if str(DEVICE) == "cuda":
    import torch
    print(f"GPU          : {torch.cuda.get_device_name(0)}")

d:\forKrishna\MOML\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device       : cuda
Project root : D:\forKrishna\MOML
GPU          : NVIDIA GeForce GTX 1650


## Configuration

Defaults give **80 evaluations** (16 Sobol-init + 64 BO) and target ≤1.5h on a small CUDA GPU even when the pymoo notebook is running concurrently and competing for GPU time.

| Knob | Default | What it does |
|---|---|---|
| `DATASET` | `fashion_mnist` | One of `fashion_mnist`, `cifar10`. Same dataset as the pymoo notebook for fair comparison. |
| `TOTAL_TRIALS` | 80 | Total evaluations. Matches pymoo's `pop_size × n_gen`. |
| `N_INIT` | 16 | Sobol quasi-random init phase. Standard ratio is ~20% of total. |
| `TRAIN_SUBSET_SIZE` | 8000 | Samples per training run. Same as pymoo for direct comparison. |
| `SEED` | 42 | Deterministic. Per-trial seed = `SEED + trial_index`. |
| `NUM_WORKERS` | 2 | DataLoader workers. Set to 0 on Windows if you hit issues. |
| `NUM_RESTARTS` | 3 | L-BFGS restarts inside `optimize_acqf`. Tuned aggressively low: search space is ~7/10 categorical so finer acqf optimization buys little. |
| `RAW_SAMPLES` | 64 | Raw Sobol candidates the acqf optimizer subsamples from. |
| `MC_SAMPLES` | 16 | Monte Carlo samples for the qLogNEHVI integral. Lower = noisier acqf, but our discrete dims dominate the noise budget anyway. |

These three acqf-only knobs do **not** affect comparability with pymoo: same search space, same 80 trials, same objectives, same CSV schema. They only reduce BoTorch's internal per-BO-step overhead (roughly 5–10× on this problem).

Set `RUN_SMOKE_TEST = True` to do a 1-2 minute sanity check before the full run.

In [2]:
# --- Configuration --------------------------------------------------------
DATASET           = "fashion_mnist"
TOTAL_TRIALS      = 80
N_INIT            = 16
TRAIN_SUBSET_SIZE = 8000
SEED              = 42
NUM_WORKERS       = 2

# Acquisition-function compute knobs. Tuned aggressively low: this problem's
# search space is dominated by categorical dims (7/10), so finer acqf
# optimization buys little signal but costs a lot of GP-posterior MC eval time.
NUM_RESTARTS      = 3
RAW_SAMPLES       = 64
MC_SAMPLES        = 16

RUN_SMOKE_TEST    = False
COMPARE_PARETO    = None              # Set to a pymoo pareto_front.csv path for joint-front
                                       # + GD comparison (last cell auto-discovers the latest).

info = get_dataset_info(DATASET)
print(f"Dataset        : {DATASET}  (classes={info['num_classes']}, native res={info['default_resolution']})")
print(f"Total trials   : {TOTAL_TRIALS} (init={N_INIT}, bo={TOTAL_TRIALS - N_INIT})")
print(f"Train subset   : {TRAIN_SUBSET_SIZE:,} samples per trial")

Dataset        : fashion_mnist  (classes=10, native res=28)
Total trials   : 80 (init=16, bo=64)
Train subset   : 8,000 samples per trial


## (Optional) Smoke test

Tiny configuration that exercises the full pipeline (Sobol init, GP fit, qLogNEHVI acqf, decoding, CSV streaming, Pareto extraction, analysis). Skip if you trust the setup.

In [3]:
if RUN_SMOKE_TEST:
    import tempfile
    with tempfile.TemporaryDirectory() as td:
        smoke_summary, smoke_dir = run_botorch_study(
            dataset_name=DATASET,
            total_trials=6,
            n_init=4,
            train_subset_size=1000,
            seed=7,
            num_workers=0,
            num_restarts=4,
            raw_samples=64,
            mc_samples=16,
            output_root=td,
        )
        smoke_metrics = analyze(smoke_dir)
        print("\nSmoke test passed.")
        print(f"  trials   : {smoke_summary['n_trials_completed']}")
        print(f"  pareto   : {smoke_summary['n_pareto_points']}")
        print(f"  hv       : {smoke_metrics['hypervolume']:,.0f}")
        print(f"  spacing  : {smoke_metrics['spacing']:.4f}")
else:
    print("Skipped (set RUN_SMOKE_TEST = True to run).")

Skipped (set RUN_SMOKE_TEST = True to run).


## Full optimization run

Streams every trial to `trials.csv` as it completes (with a `phase` column distinguishing `init` from `bo`). Stdout shows one line per trial.

Expected wall-clock with the tuned acqf knobs above: **~30-45 min solo** on a GTX 1650 with `train_subset_size=8000`, **~60-75 min** when running concurrently with the pymoo notebook. Per-BO-step overhead (GP fit + acqf optimize) drops from ~10-20s to ~1-3s — training time now dominates.

In [ ]:
summary, study_dir = run_botorch_study(
    dataset_name=DATASET,
    total_trials=TOTAL_TRIALS,
    n_init=N_INIT,
    train_subset_size=TRAIN_SUBSET_SIZE,
    seed=SEED,
    num_workers=NUM_WORKERS,
    num_restarts=NUM_RESTARTS,
    raw_samples=RAW_SAMPLES,
    mc_samples=MC_SAMPLES,
)

print("\n" + "=" * 70)
print(f"Study dir : {study_dir}")
print(f"Wall time : {summary['elapsed_seconds']/60:.1f} min")
print(f"Pareto    : {summary['n_pareto_points']} non-dominated points (over all trials)")

BoTorch qNEHVI | device=cuda | dataset=fashion_mnist | total_trials=80 (init=16, bo=64)

[phase] Sobol init
  trial 000 | acc=0.3872 | ms=  0.268 | params=    5,898 |  53.7s | init
  trial 001 | acc=0.6736 | ms=  0.324 | params=    4,762 |  56.2s | init
  trial 002 | acc=0.4752 | ms=  0.408 | params=   10,410 |  34.6s | init
  trial 003 | acc=0.8623 | ms=  0.727 | params=   20,826 |  67.4s | init
  trial 004 | acc=0.7973 | ms=  0.378 | params=   10,410 |  46.7s | init
  trial 005 | acc=0.5450 | ms=  1.146 | params=  258,762 |  61.4s | init
  trial 006 | acc=0.1084 | ms=  0.249 | params=    2,538 |  44.1s | init
  trial 007 | acc=0.8116 | ms=  0.440 | params=   10,410 |  72.0s | init
  trial 008 | acc=0.3574 | ms=  0.414 | params=    2,538 |  44.9s | init
  trial 009 | acc=0.8113 | ms=  0.691 | params=  258,762 |  57.6s | init
  trial 010 | acc=0.5828 | ms=  0.249 | params=    3,658 |  42.7s | init
  trial 011 | acc=0.6299 | ms=  0.676 | params=   28,618 |  71.6s | init
  trial 012 | ac

## Analysis — Pareto metrics, table, plots, appendix solution

Same `analyze(study_dir)` as the pymoo notebook (framework-agnostic): recomputes the non-dominated set from all trials, computes hypervolume against the project reference point `[acc=0, ms=1000, params=10M]`, the spacing metric, picks the four canonical Pareto representatives (`fast`, `small`, `accurate`, `balanced`), saves the appendix solution, writes three plots.

In [ ]:
compare_path = Path(COMPARE_PARETO) if COMPARE_PARETO else None
metrics = analyze(study_dir, compare_pareto_path=compare_path)

print(f"n_trials       : {metrics['n_trials']}")
print(f"n_pareto       : {metrics['n_pareto_points']}")
print(f"hypervolume    : {metrics['hypervolume']:,.2f}")
print(f"spacing        : {metrics['spacing']:.4f}")
print(f"acc max        : {metrics['extremes']['accuracy_max']:.4f}")
print(f"ms  min        : {metrics['extremes']['inference_ms_min']:.4f}")
print(f"params min     : {metrics['extremes']['param_count_min']:,}")
if 'comparison' in metrics:
    c = metrics['comparison']
    print("\n--- vs comparison front (pymoo) ---")
    print(f"joint pareto      : {c['joint_n_pareto']}")
    print(f"ours surviving    : {c['ours_surviving_in_joint']}")
    print(f"other surviving   : {c['other_surviving_in_joint']}")
    print(f"GD ours -> joint  : {c['gd_ours_to_joint']:.6f}")
    print(f"GD other-> joint  : {c['gd_other_to_joint']:.6f}")

### Pareto table (4 representative solutions — satisfies the assignment minimum)

In [ ]:
table_path = study_dir / "pareto_table.csv"
pareto_table = pd.read_csv(table_path)
pareto_table

### 2D panels — the three pairwise trade-offs
Filled markers = all trials. Black-outlined markers = Pareto front. Magenta diamonds appear only if `COMPARE_PARETO` is set.

In [ ]:
display(Image(filename=str(study_dir / "plot_2d_panels.png")))

### Parallel coordinates of the Pareto front
Each line is one Pareto-optimal solution; axes are normalized to `[0,1]` with **lower = better** (so accuracy is shown as `error_rate = 1-acc`).

In [ ]:
display(Image(filename=str(study_dir / "plot_parallel_coords.png")))

### 3D scatter — full objective space

In [ ]:
display(Image(filename=str(study_dir / "plot_3d_scatter.png")))

## Appendix — one fully-detailed Pareto-optimal solution
Selected as the **`balanced`** point: closest to the ideal `(max accuracy, min inference, min params)` in normalized objective space. This is what goes into the report's appendix section.

In [ ]:
with open(study_dir / "appendix_solution.json") as f:
    appendix = json.load(f)

print(f"Label              : {appendix['label']}")
print(f"Trial number       : {appendix.get('trial_number')}")
print()
print("--- Objectives ---")
for k, v in appendix['objectives'].items():
    print(f"  {k:<14s} : {v}")
print()
print("--- Decision variables ---")
for k, v in appendix['decision_variables'].items():
    print(f"  {k:<18s} : {v}")

## Compare against pymoo (run after the pymoo notebook finishes)
Once the pymoo notebook has produced `results/pymoo/<dataset>/<study>/pareto_front.csv`, set `COMPARE_PARETO` (config cell above) to that path and re-run the *Analysis* cell. The 2D panels will overlay magenta diamonds for the pymoo front, and the metrics output will include joint-front survival counts plus generational distance — the head-to-head MOO comparison the rubric asks for.

In [ ]:
# Helper: list the most recent pymoo study dir for the chosen dataset.
candidate = PROJECT_ROOT / "results" / "pymoo" / DATASET
if candidate.exists():
    runs = sorted([p for p in candidate.iterdir() if p.is_dir()])
    if runs:
        latest = runs[-1] / "pareto_front.csv"
        print(f"pymoo latest pareto: {latest}")
        print(f"\nTo enable comparison, set in the config cell:\n  COMPARE_PARETO = r\"{latest}\"")
    else:
        print(f"(no pymoo runs yet in {candidate})")
else:
    print(f"(no pymoo results for dataset={DATASET})")